In [0]:
-- AUTOMATED ALERT SYSTEM FOR HIGH-RISK STUDENTS
-- Run this query on a schedule (hourly recommended)

-- Step 1: Auto-log new high-risk students to alerts table
-- Risk score > 0.75 triggers automatic alert creation
INSERT INTO workspace.default.gold_automated_alerts
SELECT 
    CONCAT('ALERT-', CAST(student_id AS STRING), '-', DATE_FORMAT(current_timestamp(), 'yyyyMMddHHmmss')) as alert_id,
    CAST(student_id AS STRING) as student_id,
    risk_score,
    intervention_tier,
    top_3_risk_factors as top_risk_factors,
    CASE
        WHEN risk_score > 0.90 THEN 'URGENT: Immediate one-on-one counseling required. Review financial aid eligibility and academic support services.'
        WHEN risk_score > 0.80 THEN 'HIGH PRIORITY: Schedule intervention meeting within 48 hours. Assess course load and financial barriers.'
        ELSE 'MODERATE: Monitor weekly. Provide proactive academic resources and peer mentoring.'
    END as intervention_recommendation,
    current_timestamp() as alert_triggered_at,
    false as counselor_notified,
    'NEW' as alert_status
FROM workspace.default.gold_at_risk_students
WHERE risk_score > 0.75
    AND CAST(student_id AS STRING) NOT IN (
        SELECT student_id 
        FROM workspace.default.gold_automated_alerts 
        WHERE alert_status IN ('NEW', 'IN_PROGRESS')
    );

-- Step 2: Monitor query - Show newly detected alerts requiring counselor notification
SELECT 
    alert_id,
    student_id,
    risk_score,
    intervention_tier,
    top_risk_factors,
    intervention_recommendation,
    alert_triggered_at,
    counselor_notified,
    alert_status
FROM workspace.default.gold_automated_alerts
WHERE alert_status = 'NEW'
ORDER BY risk_score DESC;